# Trabajo Práctico 2 - Grupo 02

### Modelo Red Neuronal

Integrantes:

*   Bermudez, Agustin
*   Calderón, Tiago
*   Gonzalez Pautaso, Mateo
*   Moreyra, Santiago
*   Nieves, Maylen

**Dataset:** EDA aumentado 67k balanceado

**Hiperparámetros:** `lr=2e-05` | `epochs=3` | `max_length=128`

**Justificación:** LR de 2e-5 es el estándar recomendado para fine-tuning de BERT. Con el dataset balanceado de 67k, 3 épocas son suficientes para converger sin overfitting.

## 1. Instalación de dependencias


In [3]:
!pip3 install torch torchvision torchaudio
!pip3 install transformers datasets accelerate
!pip3 install spacy
!python3 -m spacy download es_core_news_sm
!pip3 install scikit-learn xgboost joblib
!pip3 install pandas numpy scipy
!pip3 install matplotlib seaborn
!pip3 install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 102.4 MB/s eta 0:00:0000:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0

In [4]:
import nltk
nltk.download("sentiwordnet")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package sentiwordnet to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package sentiwordnet is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


True

In [5]:
import os
for root, dirs, files in os.walk("/kaggle/input/"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/santiagomoreyra/common/common/preprocessing.py
/kaggle/input/datasets/santiagomoreyra/common/common/evaluation.py
/kaggle/input/datasets/santiagomoreyra/common/common/data_utils.py
/kaggle/input/datasets/santiagomoreyra/common/common/io_utils.py
/kaggle/input/datasets/santiagomoreyra/common/common/embeddings.py
/kaggle/input/datasets/santiagomoreyra/tp2-ciencia-de-datos/sample_submission.csv
/kaggle/input/datasets/santiagomoreyra/tp2-ciencia-de-datos/train_augmented_eda_balanced.csv
/kaggle/input/datasets/santiagomoreyra/tp2-ciencia-de-datos/test.csv


## 2. Imports y configuración


In [ ]:
import sys
sys.path.insert(0, "/kaggle/input/datasets/santiagomoreyra/common/")
sys.path.insert(0, "/kaggle/input/datasets/santiagomoreyra/tp2-ciencia-de-datos/")

import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import f1_score, classification_report, confusion_matrix

from common.data_utils import get_split, SEED
from common.preprocessing import clean_minimal
from common.evaluation import evaluate

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

MODEL_NAME  = "dccuchile/bert-base-spanish-wwm-cased"
MAX_LENGTH  = 128
BATCH_SIZE  = 32
NUM_EPOCHS  = 3
LR          = 2e-5
CLASS_NAMES = ["negativa", "neutra", "positiva"]

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    print("GPU: Apple Silicon (MPS)")
else:
    print("GPU: No disponible, corriendo en CPU")

## 3. Carga de datos

In [ ]:
train_df = pd.read_csv("/kaggle/input/datasets/santiagomoreyra/tp2-ciencia-de-datos/train_augmented_eda_balanced.csv")
test_df  = pd.read_csv("/kaggle/input/datasets/santiagomoreyra/tp2-ciencia-de-datos/test.csv")

print(f"Train (aumentado): {len(train_df):,} filas")
print(f"Distribucion:\n{train_df['label'].value_counts().sort_index()}")

X_train_raw, X_val_raw, y_train, y_val = get_split(train_df)

print("\nAplicando clean_minimal...")
X_train = np.array([clean_minimal(t) for t in X_train_raw])
X_val   = np.array([clean_minimal(t) for t in X_val_raw])
X_test  = np.array([clean_minimal(t) for t in test_df["text"].values])
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

## 4. Dataset y tokenización

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ResenasDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_length=128):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

print("Tokenizando...")
train_dataset = ResenasDataset(X_train, y_train, tokenizer, MAX_LENGTH)
val_dataset   = ResenasDataset(X_val,   y_val,   tokenizer, MAX_LENGTH)
test_dataset  = ResenasDataset(X_test,  None,    tokenizer, MAX_LENGTH)
print("Listo.")

## 5. Modelo y métricas


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    ignore_mismatched_sizes=True,
)
model = model.to(DEVICE)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    f1_per = f1_score(labels, preds, average=None, zero_division=0)
    return {
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_neg": float(f1_per[0]),
        "f1_neu": float(f1_per[1]),
        "f1_pos": float(f1_per[2]),
    }

## 6. Fine-tuning

In [ ]:
OUTPUT_DIR = Path("models/red_neuronal_beto_n2_eda67k_lr2e5_ep3")

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f"Iniciando fine-tuning: lr={LR} | epochs={NUM_EPOCHS} | max_length={MAX_LENGTH} | dataset=EDA_67k_balanceado")
trainer.train()

## 7. Evaluación en validación


In [ ]:
preds_output = trainer.predict(val_dataset)
y_pred = np.argmax(preds_output.predictions, axis=1)

evaluate("red_neuronal_beto_n2_eda67k_lr2e5_ep3", y_val, y_pred,
         hyperparams={
             "model": MODEL_NAME,
             "epochs": NUM_EPOCHS,
             "lr": LR,
             "max_length": MAX_LENGTH,
             "batch_size": BATCH_SIZE,
             "dataset": "eda_augmented_67k_balanced",
         })

import matplotlib.pyplot as plt

history  = trainer.state.log_history
eval_f1  = [(x["epoch"], x["eval_f1_macro"]) for x in history if "eval_f1_macro" in x]
eval_neu = [(x["epoch"], x["eval_f1_neu"])   for x in history if "eval_f1_neu"   in x]

fig, ax = plt.subplots(figsize=(8, 4))
if eval_f1:
    epochs_list, f1s = zip(*eval_f1)
    ax.plot(epochs_list, f1s, marker="o", label="F1-macro", color="#2ca02c")
    for e, f in zip(epochs_list, f1s):
        ax.annotate(f"{f:.4f}", (e, f), textcoords="offset points", xytext=(0, 8))
if eval_neu:
    epochs_list, f1s = zip(*eval_neu)
    ax.plot(epochs_list, f1s, marker="s", label="F1-neutra", color="#7f7f7f", linestyle="--")
ax.set_xlabel("Epoch"); ax.set_ylabel("F1")
ax.set_title("N2 — F1-macro y F1-neutra por época")
ax.legend(); plt.tight_layout(); plt.show()

## 8. Guardado del modelo

In [ ]:
SAVE_DIR = Path("models/red_neuronal_beto_n2_eda67k_lr2e5_ep3_final")
trainer.save_model(str(SAVE_DIR))
tokenizer.save_pretrained(str(SAVE_DIR))
print(f"Modelo guardado en {SAVE_DIR}")

## 9. Submission a Kaggle


In [ ]:
Path("submissions").mkdir(exist_ok=True)
preds_test  = trainer.predict(test_dataset)
y_test_pred = np.argmax(preds_test.predictions, axis=1)

sub = pd.DataFrame({"id": test_df["id"].values, "label": y_test_pred.astype(int)})
sub.to_csv("submissions/submission_red_neuronal_beto_n2_eda67k_lr2e5_ep3.csv", index=False)

dist = sub["label"].value_counts(normalize=True).sort_index()
print(f"Guardado: submissions/submission_red_neuronal_beto_n2_eda67k_lr2e5_ep3.csv  ({len(sub)} predicciones)")
print(f"Distribucion: {', '.join(f'clase {k}: {v:.1%}' for k, v in dist.items())}")